# Data Cleaning and Preprocessing

## Importing Libraries

In [44]:
import warnings

warnings.filterwarnings('ignore')

In [45]:
import pandas as pd
import os
import re
import nltk
import string
import numpy as np
import keras
from huggingface_hub import login
from datasets import load_dataset
import seaborn as sns
import matplotlib.pyplot as plt
import dagshub
import mlflow
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding,SpatialDropout1D
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


## Download dataset from huggingface

In [46]:
"""# Huggingface_hub Login
login()"""

"""# Loading the dataset into a pandas dataframe
df = pd.DataFrame(load_dataset("Tobi-Bueck/customer-support-tickets")['train'])

# Saving the dataset
df.to_csv("datasets/Customer_Support_Tickets.csv")"""

#df = load_dataset("Tobi-Bueck/customer-support-tickets")['train'].to_pandas()

'# Loading the dataset into a pandas dataframe\ndf = pd.DataFrame(load_dataset("Tobi-Bueck/customer-support-tickets")[\'train\'])\n\n# Saving the dataset\ndf.to_csv("datasets/Customer_Support_Tickets.csv")'

## Data Lookup

In [47]:
"""# First 5 rows
df.head()"""

'# First 5 rows\ndf.head()'

In [48]:
"""# Last 5 rows
df.tail()"""

'# Last 5 rows\ndf.tail()'

In [49]:
"""df.describe(include='all')"""

"df.describe(include='all')"

In [50]:
"""df.info()"""

'df.info()'

## Data Cleaning

### Missing values

In [51]:
"""# Calculate the percentage of missing values in each row
def per_missing_values(df):
    return df.isna().mean() * 100

per_missing_values(df)"""

'# Calculate the percentage of missing values in each row\ndef per_missing_values(df):\n    return df.isna().mean() * 100\n\nper_missing_values(df)'

### Removing rows with missing body

In [52]:
"""df = df[~df['body'].isna()]"""

"df = df[~df['body'].isna()]"

## Data Preprocessing

In [53]:
"""def clean_text(text): 
    #Make text lowercase, remove text in square brackets,remove links,remove punctuation and remove words containing numbers.
    text = text.lower() 
    text = text.replace("\\n", ' ') 
    #text = re.sub(r'\s+', ' ', text).strip() 
    #text = re.sub('\[.*?\]', '', text) 
    #text = re.sub('https?://\S+|www\.\S+', '', text) 
    #text = re.sub('<.*?>+', '', text) 
    #text = re.sub('[%s]' % re.escape(string.punctuation), '', text) 
    #text = re.sub('\w*\d\w*', '', text)
    return text 

def text_preprocessing(text): 
    #Cleaning and parsing the text.
    tokenizer = nltk.tokenize.RegexpTokenizer(r'\w+') 
    nopunc = clean_text(text) 
    tokenized_text = tokenizer.tokenize(nopunc) 
    #remove_stopwords = [w for w in tokenized_text if w not in stopwords.words('english')] 
    combined_text = ' '.join(tokenized_text) 
    return combined_text"""

'def clean_text(text): \n    #Make text lowercase, remove text in square brackets,remove links,remove punctuation and remove words containing numbers.\n    text = text.lower() \n    text = text.replace("\\n", \' \') \n    #text = re.sub(r\'\\s+\', \' \', text).strip() \n    #text = re.sub(\'\\[.*?\\]\', \'\', text) \n    #text = re.sub(\'https?://\\S+|www\\.\\S+\', \'\', text) \n    #text = re.sub(\'<.*?>+\', \'\', text) \n    #text = re.sub(\'[%s]\' % re.escape(string.punctuation), \'\', text) \n    #text = re.sub(\'\\w*\\d\\w*\', \'\', text)\n    return text \n\ndef text_preprocessing(text): \n    #Cleaning and parsing the text.\n    tokenizer = nltk.tokenize.RegexpTokenizer(r\'\\w+\') \n    nopunc = clean_text(text) \n    tokenized_text = tokenizer.tokenize(nopunc) \n    #remove_stopwords = [w for w in tokenized_text if w not in stopwords.words(\'english\')] \n    combined_text = \' \'.join(tokenized_text) \n    return combined_text'

In [54]:
"""# Combining the subject and body columns into a single column for text preprocessing

df['clean_subject'] = df['subject'].apply(str).apply(lambda x: text_preprocessing(x))
df['clean_body'] = df['body'].apply(str).apply(lambda x: text_preprocessing(x))

df['clean_sub_body'] = df['clean_subject'] + ' ' + df['clean_body']"""

"# Combining the subject and body columns into a single column for text preprocessing\n\ndf['clean_subject'] = df['subject'].apply(str).apply(lambda x: text_preprocessing(x))\ndf['clean_body'] = df['body'].apply(str).apply(lambda x: text_preprocessing(x))\n\ndf['clean_sub_body'] = df['clean_subject'] + ' ' + df['clean_body']"

In [55]:
# Combining the tag columns into a single column, making sure to drop any NaN values

"""cols = ['tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

# Combine the tag columns into a single column, making sure to drop any NaN values
df['tags'] = df[cols].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1)

# Adding tags to body and subject for better context in the model
df['clean_sub_body'] = df['clean_sub_body'] + " " + df['tags']"""

'cols = [\'tag_1\', \'tag_2\', \'tag_3\', \'tag_4\', \'tag_5\', \'tag_6\', \'tag_7\', \'tag_8\']\n\n# Combine the tag columns into a single column, making sure to drop any NaN values\ndf[\'tags\'] = df[cols].apply(lambda x: \' \'.join(x.dropna().astype(str)), axis=1)\n\n# Adding tags to body and subject for better context in the model\ndf[\'clean_sub_body\'] = df[\'clean_sub_body\'] + " " + df[\'tags\']'

In [56]:
"""# Classifying the tickets into departments based on the queue column

conditions = [
    df["queue"].str.contains(
        r"technical|api|authentication|integration|engineering|support|Service Outages and Maintenance|IT & Technology/Security Operations|IT & Technology/Network Infrastructure|IT & Technology/Software Development",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"sales|business development|account executive|General Inquiry",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"billing|payment|invoice|refund|Returns and Exchanges",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"customer success|account management|customer care|Customer Service",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"legal|compliance|privacy|gdpr|Law & Government/Government Services",
        case=False,
        na=False
    ),
    df['queue'].str.contains(
        r"hr|human resources|recruitment|talent acquisition|Human Resources",
        case=False,
        na=False
    )
]

choices = [
    "Technical",
    "Sales",
    "Billing",
    "Customer Success",
    "Legal",
    "HR"
]

df["department"] = np.select(
    conditions,
    choices,
    default="Other"
)"""

'# Classifying the tickets into departments based on the queue column\n\nconditions = [\n    df["queue"].str.contains(\n        r"technical|api|authentication|integration|engineering|support|Service Outages and Maintenance|IT & Technology/Security Operations|IT & Technology/Network Infrastructure|IT & Technology/Software Development",\n        case=False,\n        na=False\n    ),\n    df["queue"].str.contains(\n        r"sales|business development|account executive|General Inquiry",\n        case=False,\n        na=False\n    ),\n    df["queue"].str.contains(\n        r"billing|payment|invoice|refund|Returns and Exchanges",\n        case=False,\n        na=False\n    ),\n    df["queue"].str.contains(\n        r"customer success|account management|customer care|Customer Service",\n        case=False,\n        na=False\n    ),\n    df["queue"].str.contains(\n        r"legal|compliance|privacy|gdpr|Law & Government/Government Services",\n        case=False,\n        na=False\n    ),\n   

In [57]:
"""(df['department'].value_counts())"""

"(df['department'].value_counts())"

In [58]:
"""# Removing duplicate rows
df.drop_duplicates(inplace=True)"""

'# Removing duplicate rows\ndf.drop_duplicates(inplace=True)'

In [59]:
"""# Saving the cleaned dataset
df.to_csv("datasets/Customer_Support_Tickets_Cleaned.csv", index=False)"""

'# Saving the cleaned dataset\ndf.to_csv("datasets/Customer_Support_Tickets_Cleaned.csv", index=False)'

In [60]:
#df = df[df['department'] != 'Other']

In [61]:
# Loading the dataset
df = load_dataset("Tobi-Bueck/customer-support-tickets")['train'].to_pandas()

# Cleaning the text data

def clean_text(text):
    if pd.isna(text):
        return ''

    text = re.sub(r'\\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['cleaned_body'] = df['body'].apply(clean_text)
df['cleaned_subject'] = df['subject'].apply(clean_text)

# Combining subject and body for input

def combine_subject_body(row):
    subject = row['cleaned_subject']
    body = row['cleaned_body']
    if subject:
        return subject + '. ' + body
    else:
        return body


df['subject_body'] = df.apply(combine_subject_body, axis=1)


In [62]:
"""department_mapping = {
    # Support Engineering
    "Technical Support": "Support Engineering",
    "Product Support": "Support Engineering",
    "IT & Technology/Hardware Support": "Support Engineering",
    "IT & Technology/Software Development": "Support Engineering",
    "IT & Technology/Network Infrastructure": "Support Engineering",
    "IT & Technology/Security Operations": "Support Engineering",

    # Internal IT
    "IT Support": "Internal IT",

    # Operations
    "Service Outages and Maintenance": "Operations",

    # Finance
    "Billing and Payments": "Finance",

    # Sales
    "Sales and Pre-Sales": "Sales",

    # Customer Operations
    "Returns and Exchanges": "Customer Operations",
    "Customer Service": "Customer Operations",
    "General Inquiry": "Customer Operations",

    # Human Resources
    "Human Resources": "Human Resources",
    "Jobs & Education/Recruitment": "Human Resources",
}

df["department"] = (
    df["queue"]
      .map(department_mapping)
      .fillna("Others")
)"""

'department_mapping = {\n    # Support Engineering\n    "Technical Support": "Support Engineering",\n    "Product Support": "Support Engineering",\n    "IT & Technology/Hardware Support": "Support Engineering",\n    "IT & Technology/Software Development": "Support Engineering",\n    "IT & Technology/Network Infrastructure": "Support Engineering",\n    "IT & Technology/Security Operations": "Support Engineering",\n\n    # Internal IT\n    "IT Support": "Internal IT",\n\n    # Operations\n    "Service Outages and Maintenance": "Operations",\n\n    # Finance\n    "Billing and Payments": "Finance",\n\n    # Sales\n    "Sales and Pre-Sales": "Sales",\n\n    # Customer Operations\n    "Returns and Exchanges": "Customer Operations",\n    "Customer Service": "Customer Operations",\n    "General Inquiry": "Customer Operations",\n\n    # Human Resources\n    "Human Resources": "Human Resources",\n    "Jobs & Education/Recruitment": "Human Resources",\n}\n\ndf["department"] = (\n    df["queue"]\n

In [63]:
# Classifying the tickets into departments based on the queue column

conditions = [
    df["queue"].str.contains(
        r"technical|api|authentication|integration|engineering|support|Service Outages and Maintenance|IT & Technology/Security Operations|IT & Technology/Network Infrastructure|IT & Technology/Software Development",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"sales|business development|account executive|General Inquiry",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"billing|payment|invoice|refund|Returns and Exchanges",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"customer success|account management|customer care|Customer Service",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"legal|compliance|privacy|gdpr|Law & Government/Government Services",
        case=False,
        na=False
    ),
    df['queue'].str.contains(
        r"hr|human resources|recruitment|talent acquisition|Human Resources",
        case=False,
        na=False
    )
]

choices = [
    "Technical",
    "Sales",
    "Billing",
    "Customer Success",
    "Legal",
    "HR"
]

df["department"] = np.select(
    conditions,
    choices,
    default="Other"
)

In [64]:
df['department'].unique()

array(['Technical', 'Billing', 'Sales', 'Customer Success', 'HR', 'Other',
       'Legal'], dtype=object)

In [65]:
df = df[df['department'] != 'Others']
df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,cleaned_body,cleaned_subject,subject_body,department
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None,"Sehr geehrtes Support-Team, ich möchte einen g...",Wesentlicher Sicherheitsvorfall,Wesentlicher Sicherheitsvorfall. Sehr geehrtes...,Technical
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None,"Dear Customer Support Team, I am writing to re...",Account Disruption,Account Disruption. Dear Customer Support Team...,Technical
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None,"Dear Customer Support Team, I hope this messag...",Query About Smart Home System Integration Feat...,Query About Smart Home System Integration Feat...,Billing
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None,"Dear Customer Support Team, I hope this messag...",Inquiry Regarding Invoice Details,Inquiry Regarding Invoice Details. Dear Custom...,Billing
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None,"Dear Support Team, I hope this message reaches...",Question About Marketing Agency Software Compa...,Question About Marketing Agency Software Compa...,Sales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61760,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...,I would be happy to assist with the IFTTT Dock...,Problem,Technical Support,low,en,NaN,Integration,Disruption,Performance,IT,Tech Support,None,None,None,I am facing integration problems with IFTTT Do...,Assistance Needed for IFTTT Docker Integration,Assistance Needed for IFTTT Docker Integration...,Technical
61761,Bitten um Unterstützung bei der Integration,"Sehr geehrte Kundenservice, ich möchte die Int...","Sehr geehrte [Name], vielen Dank für Ihren Kon...",Change,Technical Support,medium,de,NaN,Integration,Feature,Documentation,Tech Support,None,None,None,None,"Sehr geehrte Kundenservice, ich möchte die Int...",Bitten um Unterstützung bei der Integration,Bitten um Unterstützung bei der Integration. S...,Technical
61762,None,"Hello Customer Support, I am inquiring about t...",We will send you detailed information on plans...,Request,Billing and Payments,low,en,NaN,Billing,Payment,Feature,Feedback,Sales,Lead,None,None,"Hello Customer Support, I am inquiring about t...",,"Hello Customer Support, I am inquiring about t...",Billing
61763,Hilfe bei digitalen Strategie-Problemen,Die Qualität unserer digitalen Strategie-Bearb...,Um den digitalen Strategie-Impuls zu überprüfe...,Incident,Product Support,high,de,NaN,Feedback,Performance,IT,Tech Support,None,None,None,None,Die Qualität unserer digitalen Strategie-Bearb...,Hilfe bei digitalen Strategie-Problemen,Hilfe bei digitalen Strategie-Problemen. Die Q...,Technical


In [66]:
df['department'].value_counts()

department
Technical           32261
Other               10426
Customer Success     7420
Billing              7312
Sales                2522
HR                   1204
Legal                 620
Name: count, dtype: int64

# Baseline Model

## Data Preparation

### Oversampling

In [67]:
"""from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(
    df[['subject_body']],
    df['department']
)"""

"from imblearn.over_sampling import RandomOverSampler\n\nros = RandomOverSampler(random_state=42)\nX_res, y_res = ros.fit_resample(\n    df[['subject_body']],\n    df['department']\n)"

In [68]:
"""vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=3
)

X = vectorizer.fit_transform(df['subject_body'])

le = LabelEncoder()

y = le.fit_transform(df['department'])"""

"vectorizer = TfidfVectorizer(\n    max_features=50000,\n    ngram_range=(1,2),\n    min_df=3\n)\n\nX = vectorizer.fit_transform(df['subject_body'])\n\nle = LabelEncoder()\n\ny = le.fit_transform(df['department'])"

In [69]:
le = LabelEncoder()
y = le.fit_transform(df["department"])

In [70]:
"""# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

dataset_tag = {
    'name':'RandomSampler'
}"""

"# Train-test split\nX_train, X_test, y_train, y_test = train_test_split(\n    X,\n    y,\n    test_size=0.2,\n    random_state=42,\n    stratify=y\n)\n\ndataset_tag = {\n    'name':'RandomSampler'\n}"

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    df["subject_body"],   # Raw text
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

dataset_tags = {
    'name':'Default',
    'features':'subject_body',
    'target':'department',
    'sampling': 'None'
}

## Model Training and Evaluation

### Support Vector Machine

In [72]:
"""svm_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=5000,
    class_weight='balanced'
)

svm_tags = {
    "algorithm": "Linear SVM",
    "model_type": "LinearSVC"
}"""

'svm_model = LinearSVC(\n    C=1.0,\n    random_state=42,\n    max_iter=5000,\n    class_weight=\'balanced\'\n)\n\nsvm_tags = {\n    "algorithm": "Linear SVM",\n    "model_type": "LinearSVC"\n}'

In [73]:
svm_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            min_df=3
        )
    ),
    (
        "classifier",
        LinearSVC(
            C=1.0,
            random_state=42,
            max_iter=5000,
            class_weight="balanced"
        )
    )
])

svm_tags = {
    "algorithm": "Linear SVM",
    "model_type": "LinearSVC",
    "class_weight":'balanced'
}

### Logistic Regression

In [74]:
"""# Model
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_tags = {
    'max_iter': 1000,
    'class_weight': 'balanced'
}"""

'# Model\nlr_model = LogisticRegression(\n    max_iter=1000,\n    class_weight="balanced",\n    random_state=42\n)\n\nlr_tags = {\n    \'max_iter\': 1000,\n    \'class_weight\': \'balanced\'\n}'

### Naive Bayes

In [75]:
"""nb_model = MultinomialNB()"""

'nb_model = MultinomialNB()'

### Evaluation Plots

In [76]:
# ---------------------------------------------
# ROC Curve Function
# ---------------------------------------------

def create_roc_curve(model, model_name, X_test, y_test, labels):
    
    n_classes = len(labels)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)

    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    elif isinstance(model, tf.keras.Model):
        y_score = model.predict(X_test)

    else:
        raise ValueError(
            f"{type(model).__name__} does not support "
            "predict_proba or decision_function."
        )

    # ======================
    # Binary Classification
    # ======================
    if n_classes == 2:

        if y_score.ndim > 1:
            y_score_binary = y_score[:, 1]
        else:
            y_score_binary = y_score

        fpr, tpr, _ = roc_curve(
            y_test,
            y_score_binary
        )

        roc_auc = auc(fpr, tpr)

        fig, ax = plt.subplots(figsize=(7, 5))

        ax.plot(
            fpr,
            tpr,
            lw=2,
            label=f"AUC = {roc_auc:.3f}"
        )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Multiclass Classification
    # ======================
    else:

        # One-hot encode y_test
        y_test_bin = label_binarize(
            y_test,
            classes=np.arange(n_classes)
        )

        roc_auc = roc_auc_score(
            y_test_bin,
            y_score,
            multi_class="ovr",
            average="weighted"
        )

        fig, ax = plt.subplots(figsize=(8, 6))

        for i in range(n_classes):

            fpr, tpr, _ = roc_curve(
                y_test_bin[:, i],
                y_score[:, i]
            )

            class_auc = auc(fpr, tpr)

            ax.plot(
                fpr,
                tpr,
                lw=1.5,
                label=f"{labels[i]} (AUC={class_auc:.3f})"
            )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Common Plot Formatting
    # ======================
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve — {model_name}")

    ax.legend(
        loc="lower right",
        fontsize=8
    )

    plt.tight_layout()

    # Save plot
    save_dir = os.path.join(
        "Images",
        "ROC_Curves"
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    save_path = os.path.join(
        save_dir,
        f"{model_name}.png"
    )

    fig.savefig(
        save_path,
        dpi=150
    )

    plt.close(fig)

    print(
        f"ROC Curve saved → {save_path} "
        f"(AUC: {roc_auc:.3f})"
    )

    return save_path, roc_auc

# ---------------------------------------------
# Confusion Matrix Function
# ---------------------------------------------

def create_confusion_matrix(model_name, y_true, y_pred, class_names, path="Images"):
    
    cm = confusion_matrix(y_true, y_pred)

    save_dir = os.path.join(path, "Confusion Matrix")
    os.makedirs(save_dir, exist_ok=True)

    safe_name = model_name.replace(" ", "_")
    save_path = os.path.join(save_dir, f"{safe_name}.png")

    plt.figure(figsize=(8, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

    print(f"Confusion Matrix saved → {save_path}")

    return save_path

### Training and Evaluation

In [77]:
models = [
    (
        "Linear SVM",
        svm_pipeline,
        (X_train, y_train),
        (X_test, y_test),
        {}
    )    
]

"""(
    "Logistic Regression",
    lr_model,
    (X_train, y_train),
    (X_test, y_test),
    lr_tags
),
(
    "Multinomial Naive Bayes",
    nb_model,
    (X_train, y_train),
    (X_test, y_test),
    {}
)"""

'(\n    "Logistic Regression",\n    lr_model,\n    (X_train, y_train),\n    (X_test, y_test),\n    lr_tags\n),\n(\n    "Multinomial Naive Bayes",\n    nb_model,\n    (X_train, y_train),\n    (X_test, y_test),\n    {}\n)'

In [78]:
model_reports = []
model_roc_paths = []
model_roc_auc_scores = []
model_cm_path = []

for model_name, model, train_set, test_set, tags in models:

    # Unpack train and test sets
    print(f"{dataset_tags['name']} data used")
    X_train, y_train = train_set
    X_test, y_test = test_set

    # Model Training
    model.fit(X_train, y_train)

    print(f"{model_name} trained successfully!")

    # Predictions and Evaluation
    y_pred = model.predict(X_test)

    labels = le.classes_
    report = classification_report(
        y_test, 
        y_pred, 
        target_names=labels, 
        output_dict=True
    )
    model_reports.append(report)

    # ROC Curve
    roc_path, roc_auc = create_roc_curve(
        model, 
        model_name, 
        X_test, 
        y_test, 
        labels
    )

    model_roc_paths.append(roc_path)
    model_roc_auc_scores.append(roc_auc)
    
    print(f"{model_name} ROC AUC saved successfully!")

    # Confusion Matrix
    cm_path = create_confusion_matrix(
        model_name=model_name,
        y_true=y_test,
        y_pred=y_pred,
        class_names=le.classes_
    )
    model_cm_path.append(cm_path)

    print(f"{model_name} Confusion Matrix saved successfully!")
    print("--------------------------------------------------------")

Default data used
Linear SVM trained successfully!
ROC Curve saved → Images\ROC_Curves\Linear SVM.png (AUC: 0.935)
Linear SVM ROC AUC saved successfully!
Confusion Matrix saved → Images\Confusion Matrix\Linear_SVM.png
Linear SVM Confusion Matrix saved successfully!
--------------------------------------------------------


### Model Tracking in Dagshub

In [ ]:
import math
dagshub.init(repo_owner='JS-Tharun', repo_name='Customer-Support-Automation', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    X_train, y_train = element[2]
    X_test, y_test = element[3]
    model_tags = element[4]

    model_report = model_reports[i]
    model_roc_auc = model_roc_auc_scores[i]
    roc_path = model_roc_paths[i]    

    with mlflow.start_run(run_name=model_name):

        # Log model parameters
        mlflow.set_tags(tags)

        # Log Model
        mlflow.sklearn.log_model(
            model, 
            model_name
        )
        mlflow.log_artifact(le)

        # Log Metrics
        def sanitize_mlflow_key(key: str) -> str:
            """Replace characters invalid in MLflow metric keys."""
            key = key.replace(" ", "_")   # 'weighted avg' → 'weighted_avg'
            key = key.replace("-", "_")   # 'f1-score'     → 'f1_score'
            return re.sub(r"[^\w./\-]", "_", key)  # catch-all for anything else

        mlflow_metrics = {}
        for label, metrics in model_report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    if metric_name != "support":
                        raw_key = f"{label}_{metric_name}"
                        key = sanitize_mlflow_key(raw_key)  # ← sanitize here
                        mlflow_metrics[key] = float(value)
            else:
                key = sanitize_mlflow_key(label)            # ← and here
                mlflow_metrics[key] = float(metrics)

        

        for k, v in mlflow_metrics.items():
            if math.isnan(v):
                print("NaN:", k)
            if math.isinf(v):
                print("Inf:", k)

        print(model_name)
        print(mlflow_metrics)      
        mlflow.log_metrics(mlflow_metrics)

        
        # Log ROC AUC metric
        mlflow_metrics["roc_auc"] = float(model_roc_auc)   # ← AUC as a metric
        mlflow.log_metrics(mlflow_metrics)
        print("Metrics Logged")

        # Log ROC AUC curve image
        mlflow.log_artifact(roc_path, artifact_path="plots/ROC AUC Curve")

        # Log Confusion matrix image
        mlflow.log_artifact(
            model_cm_path[i], 
            artifact_path="plots/Confusion Matrix"
        )


Accessing as JS-Tharun

Initialized MLflow to track repo "JS-Tharun/Customer-Support-Automation"

Repository JS-Tharun/Customer-Support-Automation initialized!

2026/06/12 17:31:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/12 17:31:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear SVM
{'Billing_precision': 0.7758389261744967, 'Billing_recall': 0.7901572112098428, 'Billing_f1_score': 0.7829326109041652, 'Customer_Success_precision': 0.6205507051712559, 'Customer_Success_recall': 0.6226415094339622, 'Customer_Success_f1_score': 0.6215943491422805, 'HR_precision': 0.8717948717948718, 'HR_recall': 0.7053941908713693, 'HR_f1_score': 0.7798165137614679, 'Legal_precision': 0.9421487603305785, 'Legal_recall': 0.9193548387096774, 'Legal_f1_score': 0.9306122448979591, 'Other_precision': 0.9448998178506375, 'Other_recall': 0.9952038369304557, 'Other_f1_score': 0.9693996729736043, 'Sales_precision': 0.6575963718820862, 'Sales_recall': 0.5753968253968254, 'Sales_f1_score': 0.6137566137566137, 'Technical_precision': 0.8735399470487463, 'Technical_recall': 0.8693428394296342, 'Technical_f1_score': 0.8714363396255729, 'accuracy': 0.8368817291346232, 'macro_avg_precision': 0.8123384857503818, 'macro_avg_recall': 0.7824987502831096, 'macro_avg_f1_score': 0.7956497635802376

# Neural Network Models

## Dataset Preparation

In [425]:

# Word Tokenization
MAX_WORDS = 20000
MAX_SEQ_LEN = 200
EMBEDDING_DIM = 100
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X_res['subject_body'].values)
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

Found 27888 unique tokens.


### Oversampling

In [426]:
"""from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_res, y_res = ros.fit_resample(
    df[['subject_body']],
    df['queue']
)"""

"from imblearn.over_sampling import RandomOverSampler\n\nros = RandomOverSampler(random_state=42)\nX_res, y_res = ros.fit_resample(\n    df[['subject_body']],\n    df['queue']\n)"

In [427]:
#df_us = pd.read_csv('datasets/Undersampled.csv')

# Paddding the sequences to ensure uniform input length for the model
X = tokenizer.texts_to_sequences(X_res['subject_body'].values)
X = pad_sequences(X, maxlen=MAX_SEQ_LEN)
print('Shape of data tensor:', X.shape)

le = LabelEncoder()

Y = le.fit_transform(y_res)

print('Shape of label tensor:', Y)

Shape of data tensor: (170576, 200)
Shape of label tensor: [6 6 0 ... 5 5 5]


### Train Test Split

In [428]:
#Train test split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.10, random_state = 42)
print(X_train.shape,Y_train.shape)
print(X_test.shape,Y_test.shape)

(153518, 200) (153518,)
(17058, 200) (17058,)


## Model Pipeline

### LSTM Pipeline

In [429]:
# LSTM

lstm_tags = {
    "frameword": "tensorflow.keras",
    "model_type": "LSTM"
}

lstm_train_params = {
    'optimizer': 'adam',
    'loss': 'sparse_categorical_crossentropy',
    'metrics': ['accuracy'],
    'epochs': 30,
    'batch_size': 256
}

lstm_model_params = {
    "max_words": MAX_WORDS,
    "embedding_dim": EMBEDDING_DIM,
    "input_length": X.shape[1],
    "lstm_units": 100,
    "dropout": 0.2,
    #"recurrent_dropout": 0.2
}


lstm_model = Sequential([
    Embedding(
        lstm_model_params["max_words"], 
        lstm_model_params["embedding_dim"], 
        input_length=lstm_model_params["input_length"]
    ),
    SpatialDropout1D(
        lstm_model_params["dropout"]
    ),
    LSTM(
        lstm_model_params["lstm_units"], 
        dropout=lstm_model_params["dropout"], 
        #recurrent_dropout=lstm_model_params["recurrent_dropout"]
    ),
    Dense(
        len(le.classes_), 
        activation='softmax'
    )
])

## Model Training

In [430]:
models = [
    (
        "LSTM",
        lstm_model,
        (X_train, Y_train),
        (X_test, Y_test),
        lstm_model_params,
        lstm_train_params,
        lstm_tags
    )
]

In [431]:
model_reports = []
model_history = []
model_roc_paths = []
model_roc_auc_scores = []
model_cm_path = []

for model_name, model, train_set, test_set, model_params, train_params, tags in models:

    X_train, Y_train = train_set
    X_test, Y_test = test_set
    

    # Model Training
    model.compile(
        optimizer=train_params['optimizer'],
        loss=train_params['loss'],
        metrics=train_params['metrics']
    )


    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        min_delta=0.001
    )


    history = model.fit(
        X_train,
        Y_train,
        epochs=train_params['epochs'],
        batch_size=train_params['batch_size'],
        callbacks=[early_stop]
    )
    model_history.append(history)


    # Model Evaluation
    predictions = model.predict(X_test)
    y_pred = np.argmax(predictions, axis=1)


    # Saving Keras Model Locally
    model_save_dir = "saved_models"
    os.makedirs(model_save_dir, exist_ok=True)
    model_save_path = os.path.join(model_save_dir, f"{model_name}.keras")
    model.save(model_save_path)

    print(f"{model_name} saved to {model_save_path}")


    # Classification Report
    labels = le.classes_
    report = classification_report(
        Y_test,
        y_pred,
        target_names=labels,
        output_dict= True
    )
    model_reports.append(report)


    # ROC Curve
    roc_path, roc_auc = create_roc_curve(model, model_name, X_test, Y_test, labels)
    model_roc_paths.append(roc_path)
    model_roc_auc_scores.append(roc_auc)
    print(f"{model_name} ROC AUC saved successfully!")


    # Confusion Matrix
    cm_path = create_confusion_matrix(
        model_name=model_name,
        y_true=Y_test,
        y_pred=y_pred,
        class_names=le.classes_
    )
    model_cm_path.append(cm_path)
    print(f"{model_name} Confusion Matrix saved successfully!")
    print("--------------------------------------------------------")

Epoch 1/30
600/600 [==============================] - ETA: 0s - loss: 1.0741 - accuracy: 0.5809WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


600/600 [==============================] - 35s 56ms/step - loss: 1.0741 - accuracy: 0.5809
Epoch 2/30
599/600 [============================>.] - ETA: 0s - loss: 0.5862 - accuracy: 0.7715WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


600/600 [==============================] - 32s 54ms/step - loss: 0.5863 - accuracy: 0.7714
Epoch 3/30
600/600 [==============================] - ETA: 0s - loss: 0.4406 - accuracy: 0.8316WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


600/600 [==============================] - 20s 34ms/step - loss: 0.4406 - accuracy: 0.8316
Epoch 4/30
600/600 [==============================] - ETA: 0s - loss: 0.3591 - accuracy: 0.8651WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


600/600 [==============================] - 18s 31ms/step - loss: 0.3591 - accuracy: 0.8651
Epoch 5/30
599/600 [============================>.] - ETA: 0s - loss: 0.3029 - accuracy: 0.8877WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


...

600/600 [==============================] - 17s 29ms/step - loss: 0.0573 - accuracy: 0.9807
Epoch 27/30
599/600 [============================>.] - ETA: 0s - loss: 0.0528 - accuracy: 0.9818WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


600/600 [==============================] - 17s 28ms/step - loss: 0.0528 - accuracy: 0.9818
Epoch 28/30
599/600 [============================>.] - ETA: 0s - loss: 0.0520 - accuracy: 0.9819WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


600/600 [==============================] - 17s 28ms/step - loss: 0.0520 - accuracy: 0.9819
Epoch 29/30
599/600 [============================>.] - ETA: 0s - loss: 0.0486 - accuracy: 0.9834WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


600/600 [==============================] - 17s 28ms/step - loss: 0.0486 - accuracy: 0.9834
Epoch 30/30
599/600 [============================>.] - ETA: 0s - loss: 0.0484 - accuracy: 0.9836WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


534/534 [==============================] - 6s 10ms/step
LSTM saved to saved_models\LSTM.keras
534/534 [==============================] - 5s 9ms/step
ROC Curve saved → Images\ROC_Curves\LSTM.png (AUC: 0.996)
LSTM ROC AUC saved successfully!
Confusion Matrix saved → Images\Confusion Matrix\LSTM.png
LSTM Confusion Matrix saved successfully!
--------------------------------------------------------


In [432]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Customer-Support-Automation', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    X_train, Y_train = element[2]
    X_test, Y_test = element[3]
    model_params = element[4]
    train_params = element[5]
    tags = element[6]

    model_report = model_reports[i]
    history = model_history[i]
    model_roc_auc = model_roc_auc_scores[i]
    roc_path = model_roc_paths[i]

    with mlflow.start_run(run_name=model_name):

        # ---------------------------------------------------
        # Logging Parameters and Tags
        # ---------------------------------------------------

        mlflow.set_tags(tags)

        mlflow.log_params(model_params)
        mlflow.log_params(train_params)

        print(f"{model_name} parameters and tags logged!")

        #-----------------------------------------------------
        # Model training and validation performance tracking
        #-----------------------------------------------------

        for epoch in range(len(history.history['loss'])):
            mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
            mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)

        print(f"{model_name} training and validation performance logged!")

        #------------------------------------------------------
        # Log Model
        #------------------------------------------------------

        model_path = os.path.join("saved_models", f"{model_name}.keras")

        if os.path.exists(model_path):
            keras_model = keras.models.load_model(model_path)
            model_info = mlflow.keras.log_model(
                keras_model,
                artifact_path=model_name
            )
            print(f"{model_name} model logged!")

        else:
            print(f"{model_name} model file not found at {model_path}!")


        #------------------------------------------------------
        # Model testing performance tracking
        #------------------------------------------------------

        # Log Metrics
        def sanitize_mlflow_key(key: str) -> str:
            """Replace characters invalid in MLflow metric keys."""
            key = key.replace(" ", "_")   # 'weighted avg' → 'weighted_avg'
            key = key.replace("-", "_")   # 'f1-score'     → 'f1_score'
            return re.sub(r"[^\w./\-]", "_", key)  # catch-all for anything else

        mlflow_metrics = {}
        for label, metrics in model_report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    if metric_name != "support":
                        raw_key = f"{label}_{metric_name}"
                        key = sanitize_mlflow_key(raw_key)  # ← sanitize here
                        mlflow_metrics[key] = float(value)
            else:
                key = sanitize_mlflow_key(label)            # ← and here
                mlflow_metrics[key] = float(metrics)

        print(f"{model_name} metrics logged")

        # Log ROC AUC metric
        mlflow_metrics["roc_auc"] = float(model_roc_auc)   # ← AUC as a metric
        mlflow.log_metrics(mlflow_metrics)
        print("Metrics Logged")

        # Log ROC AUC curve image
        mlflow.log_artifact(roc_path, artifact_path="plots/ROC AUC Curve")

        # Log Confusion matrix image
        mlflow.log_artifact(model_cm_path[i], artifact_path="plots/Confusion Matrix")

Initialized MLflow to track repo "JS-Tharun/Customer-Support-Automation"

Repository JS-Tharun/Customer-Support-Automation initialized!

LSTM parameters and tags logged!
LSTM training and validation performance logged!


2026/06/10 16:12:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 16:12:31 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpd9yy25f2\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpd9yy25f2\model\data\model\assets
2026/06/10 16:12:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


LSTM model logged!
LSTM metrics logged
Metrics Logged
🏃 View run LSTM at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0/runs/88c17860f167465991eafcbe7335586a
🧪 View experiment at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0
